# Preprocesamiento de Datos - Clustering de Usuarios

Este notebook realiza la **transformación del dataset de transacciones a dataset de usuarios** para el proyecto de clustering.

## Objetivos del preprocesamiento:

1. ✅ **Feature Engineering**: Generar variables RFM (Recency, Frequency, Monetary) y comportamiento
2. ✅ **Limpieza de outliers**: Detectar y eliminar/ajustar usuarios anómalos con Isolation Forest
3. ✅ **Escalado**: Normalizar features con RobustScaler (resistente a outliers)
4. ✅ **Encoding**: Codificar variables categóricas con One-Hot Encoding
5. ✅ **Reducción de dimensionalidad**: Aplicar PCA manteniendo ≥60% varianza explicada

## Dataset de entrada:
- **Archivo**: `data/interim/interim_ProyClustering/data_sanitized.csv`
- **Estructura**: Dataset de transacciones limpio (1 fila = 1 transacción)
- **Características**: 0 nulos, 0 duplicados, outliers capados, cancelaciones conservadas

## Dataset de salida:
- **Archivo**: `data/processed/data_clustering_pca.csv`
- **Estructura**: Dataset de usuarios (1 fila = 1 usuario/CustomerID)
- **Características**: Features escaladas, reducidas con PCA, listas para clustering

---
## 0. Imports, Configuración y Carga del Dataset Limpio

Preparamos el entorno de trabajo:
- Importamos todas las librerías necesarias para preprocesamiento y clustering
- Configuramos la estética visual global
- Definimos las rutas del proyecto
- Cargamos el dataset limpio de transacciones
- Verificamos que el dataset está correcto (dimensiones, nulos, duplicados)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 0. IMPORTS Y CONFIGURACIÓN
# ══════════════════════════════════════════════════════════════════════════════

# ── Librerías básicas ─────────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Visualización ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# ── Preprocesamiento y escalado ───────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# ── Reducción de dimensionalidad ──────────────────────────────────────────────
from sklearn.decomposition import PCA

# ── Detección de anomalías ────────────────────────────────────────────────────
from sklearn.ensemble import IsolationForest

# ── Persistencia de modelos ───────────────────────────────────────────────────
import joblib

# ── Configuración visual global ───────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10
%matplotlib inline

# ── Configuración de pandas ───────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ Librerías importadas correctamente')
print(f'\n📦 Versiones:')
print(f'   - pandas: {pd.__version__}')
print(f'   - numpy: {np.__version__}')
print(f'   - matplotlib: {plt.matplotlib.__version__}')
print(f'   - seaborn: {sns.__version__}')

In [ ]:
# ── Rutas del proyecto ───────────────────────────────────────────────────────

# Ruta del dataset limpio (input)
RUTA_DATA_LIMPIA = '../../../data/interim/interim_ProyClustering/data_sanitized.csv'

# Rutas de salida
RUTA_PROCESSED = '../../../data/processed/'
RUTA_GRAFICOS  = '../../../graphics/gr_clustering/preprocesing/'
RUTA_MODELS    = '../../../models/'

# Crear directorios si no existen
os.makedirs(RUTA_PROCESSED, exist_ok=True)
os.makedirs(RUTA_GRAFICOS, exist_ok=True)
os.makedirs(RUTA_MODELS, exist_ok=True)

print('✅ Rutas configuradas correctamente')
print(f'\n📂 Estructura de directorios:')
print(f'   Input:')
print(f'   - Dataset limpio      : {RUTA_DATA_LIMPIA}')
print(f'\n   Output:')
print(f'   - Datos procesados    : {RUTA_PROCESSED}')
print(f'   - Gráficos            : {RUTA_GRAFICOS}')
print(f'   - Modelos (scalers)   : {RUTA_MODELS}')

In [ ]:
# ── Carga del dataset limpio ─────────────────────────────────────────────────

print('='*80)
print('  CARGA DEL DATASET LIMPIO')
print('='*80)
print()
print(f'📂 Cargando desde: {RUTA_DATA_LIMPIA}')
print()

# Cargar el dataset
df_transacciones = pd.read_csv(RUTA_DATA_LIMPIA)

# Convertir columnas de fecha a datetime
df_transacciones['InvoiceDate'] = pd.to_datetime(df_transacciones['InvoiceDate'])
df_transacciones['Fecha'] = pd.to_datetime(df_transacciones['Fecha'])
df_transacciones['Mes'] = df_transacciones['Mes'].astype(str)

print(f'✅ Dataset cargado correctamente')
print()
print(f'{'─'*80}')
print(f'  DIMENSIONES DEL DATASET')
print(f'{'─'*80}')
print(f'  Filas (transacciones)     : {len(df_transacciones):>15,}')
print(f'  Columnas                  : {len(df_transacciones.columns):>15}')
print(f'  Tamaño en memoria         : {df_transacciones.memory_usage(deep=True).sum() / 1024**2:>15,.2f} MB')
print(f'{'─'*80}')
print()
print(f'  COLUMNAS DISPONIBLES ({len(df_transacciones.columns)}):')
for i, col in enumerate(df_transacciones.columns, 1):
    dtype = str(df_transacciones[col].dtype)
    print(f'    {i:>2}. {col:<20} ({dtype})')
print()
print('='*80)

In [ ]:
# ── Verificación de calidad del dataset ──────────────────────────────────────

print('='*80)
print('  VERIFICACIÓN DE CALIDAD DEL DATASET')
print('='*80)
print()

# ── Verificación 1: Valores nulos ─────────────────────────────────────────────
nulos_total = df_transacciones.isnull().sum().sum()
nulos_por_columna = df_transacciones.isnull().sum()

print(f'  ✓ VERIFICACIÓN 1: VALORES NULOS')
print(f'  {'─'*78}')
print(f'     Total valores nulos       : {nulos_total:>10,}')
if nulos_total > 0:
    print(f'     ⚠️  ATENCIÓN: Hay valores nulos en el dataset')
    print(f'\n     Columnas con nulos:')
    for col, count in nulos_por_columna[nulos_por_columna > 0].items():
        pct = (count / len(df_transacciones) * 100)
        print(f'       - {col:<20} : {count:>8,} ({pct:>6.2f}%)')
else:
    print(f'     ✅ OK: No hay valores nulos')
print()

# ── Verificación 2: Duplicados ────────────────────────────────────────────────
duplicados = df_transacciones.duplicated().sum()

print(f'  ✓ VERIFICACIÓN 2: DUPLICADOS EXACTOS')
print(f'  {'─'*78}')
print(f'     Filas duplicadas          : {duplicados:>10,}')
if duplicados > 0:
    print(f'     ⚠️  ATENCIÓN: Hay {duplicados:,} filas duplicadas')
else:
    print(f'     ✅ OK: No hay duplicados')
print()

# ── Verificación 3: CustomerID únicos ─────────────────────────────────────────
num_usuarios = df_transacciones['CustomerID'].nunique()
customerid_nulos = df_transacciones['CustomerID'].isnull().sum()

print(f'  ✓ VERIFICACIÓN 3: CUSTOMERID (crítico para clustering)')
print(f'  {'─'*78}')
print(f'     Usuarios únicos (CustomerID) : {num_usuarios:>10,}')
print(f'     CustomerID nulos             : {customerid_nulos:>10,}')
if customerid_nulos > 0:
    print(f'     ❌ ERROR: Hay CustomerID nulos → no se puede hacer clustering')
else:
    print(f'     ✅ OK: Todos los registros tienen CustomerID')
print()

# ── Verificación 4: Distribución básica ───────────────────────────────────────
num_productos = df_transacciones['StockCode'].nunique()
num_facturas = df_transacciones['InvoiceNo'].nunique()
num_paises = df_transacciones['Country'].nunique()

print(f'  ✓ VERIFICACIÓN 4: ENTIDADES ÚNICAS')
print(f'  {'─'*78}')
print(f'     Usuarios (CustomerID)     : {num_usuarios:>10,}')
print(f'     Productos (StockCode)     : {num_productos:>10,}')
print(f'     Facturas (InvoiceNo)      : {num_facturas:>10,}')
print(f'     Países                    : {num_paises:>10,}')
print()

# ── Verificación 5: Rango temporal ────────────────────────────────────────────
fecha_min = df_transacciones['InvoiceDate'].min()
fecha_max = df_transacciones['InvoiceDate'].max()
dias_totales = (fecha_max - fecha_min).days

print(f'  ✓ VERIFICACIÓN 5: COBERTURA TEMPORAL')
print(f'  {'─'*78}')
print(f'     Fecha inicio              : {fecha_min.strftime("%Y-%m-%d")}')
print(f'     Fecha fin                 : {fecha_max.strftime("%Y-%m-%d")}')
print(f'     Duración                  : {dias_totales} días')
print()

# ── Resumen de validación ─────────────────────────────────────────────────────
print(f'{'='*80}')
errores = []
if nulos_total > 0:
    errores.append('Valores nulos detectados')
if duplicados > 0:
    errores.append('Duplicados detectados')
if customerid_nulos > 0:
    errores.append('CustomerID nulos (CRÍTICO)')

if len(errores) == 0:
    print(f'  ✅ DATASET VALIDADO CORRECTAMENTE')
    print(f'  {'─'*78}')
    print(f'     El dataset está limpio y listo para preprocesamiento')
    print(f'     Próximo paso: Generar features de usuario (RFM + comportamiento)')
else:
    print(f'  ❌ ERRORES DETECTADOS EN EL DATASET')
    print(f'  {'─'*78}')
    for error in errores:
        print(f'     • {error}')
    print(f'\n     ⚠️  Revisar el notebook de limpieza antes de continuar')

print(f'{'='*80}')

In [ ]:
# ── Vista previa del dataset ─────────────────────────────────────────────────

print('\n👁️  VISTA PREVIA DEL DATASET (primeras 10 filas):')
print('='*80)
display(df_transacciones.head(10))

print('\n📊 ESTADÍSTICAS DESCRIPTIVAS (columnas numéricas):')
print('='*80)
display(df_transacciones.describe())

print('\n📋 INFORMACIÓN DEL DATAFRAME:')
print('='*80)
df_transacciones.info()

---

## ✅ PASO 0 COMPLETADO

**Dataset de transacciones cargado y verificado correctamente**

### Resumen:
- ✅ Librerías importadas (pandas, sklearn, PCA, IsolationForest)
- ✅ Rutas configuradas
- ✅ Dataset limpio cargado: `data_sanitized.csv`
- ✅ Verificaciones de calidad realizadas (0 nulos, 0 duplicados, CustomerID completo)

### Dimensiones actuales:
- **Filas**: ~400,000 transacciones
- **Columnas**: 13 columnas (InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country, Fecha, Mes, DiaSemana, EsCancelacion, TotalPrice)
- **Usuarios únicos**: ~4,400 CustomerID

---

## 🎯 SIGUIENTE PASO: PASO 4 - GENERACIÓN DE FEATURES

**Objetivo**: Transformar dataset de transacciones → dataset de usuarios

En el siguiente paso crearemos:
1. **Variables RFM** (Recency, Frequency, Monetary)
2. **Variables de comportamiento** (ticket promedio, diversidad productos, gasto mensual)
3. **Variables de cancelación** (tasa devolución, ratio devuelto)
4. **Variables temporales** (día preferido, fin de semana)
5. **Variable geográfica** (país)

**Resultado esperado**: DataFrame con ~4,400 filas (1 fila por usuario) y 10-15 features